# Household Electricity Bill Increase Classification — Preprocessing

This notebook prepares the household electricity survey data for the K-NN and MLP classification experiments.

**Personal contribution:** Data preprocessing

Pipeline:
1. Load the raw survey data
2. Remove unsuitable/high-cardinality or text-only fields
3. Map categorical responses to numeric representations
4. Handle selected missing values
5. Construct the binary target
6. Assemble the final feature matrix
7. Save the processed dataset

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Load Data

The public repository version uses a local project path rather than the original internship GitHub URL.

In [5]:
df = pd.read_csv("../data/raw_household_survey.csv")
df.head()

,Unnamed: 0,x1,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,x17,x18,x19,x20,x21,x22,x23,x24,x35,x36,x37,x38,x39,x40,x41,x42,x43,x44,x45,x46,x47,x48,x49,x50,x51,x52,x53,x54,x55,x56,x57,x58,x60,x62
0,0,Iya,Rumah,4,Sejuk,> 2200 VA,Tidak,lebih dari 20,Tidak ada,Tidak ada,1,Tidak ada,1,Tidak ada,Tidak ada,Tidak ada,Tidak ada,3,1,1,4,Tidak ada,lebih dari 5,1,2 - 4 jam,4 - 8 jam,> 8 jam,> 12 jam,2 - 4 jam,8 - 12 jam,> 4 Kali,> 4 kali,Ya,Ya,Tidak,Tidak,Tidak,Ya,Ya,9-12 Jam,> 12 Jam,3-8 Jam,3-8 Jam,3-8 Jam,9-12 Jam,TIdak,Tidak mempunyai kendaraan listrik,Rp. 1.000.001 - Rp 1.500.000,"Januari, Juni, Juli, Desember",3.0
1,1,Iya,Rumah,4,Sejuk,> 2200 VA,Ya,lebih dari 20,2,1,1,1,Tidak ada,Tidak ada,2,3,2,2,2,2,4,Tidak ada,2,2,4 - 8 jam,4 - 8 jam,4 - 8 jam,8 - 12 jam,8 - 12 jam,4 - 8 jam,2 kali,2 kali,Ya,Ya,Ya,Tidak,Ya,Ya,Tidak,> 12 Jam,3-8 Jam,9-12 Jam,9-12 Jam,3-8 Jam,9-12 Jam,Ya,Tidak mempunyai kendaraan listrik,Rp. 1.500.001 - Rp 2.000.000,Tidak Ada perbedaan signifikan,2.0
2,2,Iya,Kos,1,Sejuk,900 VA,Ya,1-5,1,Tidak ada,Tidak ada,1,Tidak ada,Tidak ada,1,1,1,Tidak ada,1,1,1,Tidak ada,Tidak ada,Tidak ada,Tidak punya TV,Tidak punya TV,4 - 8 jam,8 - 12 jam,4 - 8 jam,8 - 12 jam,Tidak punya mesin cuci,Tidak punya setrika,Tidak,Tidak,Tidak,Tidak,Tidak,Tidak,Tidak,9-12 Jam,<2 Jam,<2 Jam,<2 Jam,<2 Jam,<2 Jam,TIdak,Tidak mempunyai kendaraan listrik,< Rp. 500.000,"November, Desember",4.0
3,3,Iya,Rumah,> 5,Panas,> 2200 VA,Ya,lebih dari 20,1,1,1,1,1,Tidak ada,Tidak ada,Tidak ada,Tidak ada,1,2,1,4,Tidak ada,2,1,< 2 jam,< 2 jam,> 8 jam,> 12 jam,Tidak punya komputer,Tidak punya komputer,3 kali,1 kali,Ya,Ya,Tidak,Tidak,Tidak,Tidak,Tidak,9-12 Jam,<2 Jam,<2 Jam,<2 Jam,3-8 Jam,9-12 Jam,Ya,Tidak mempunyai kendaraan listrik,> Rp 2.000.000,Tidak Ada perbedaan signifikan,3.0
4,4,Iya,Rumah,5,Panas,900 VA,Ya,lebih dari 20,1,Tidak ada,1,Tidak ada,Tidak ada,2,1,Tidak ada,Tidak ada,lebih dari 5,Tidak ada,2,lebih dari 5,Tidak ada,2,Tidak ada,< 2 jam,< 2 jam,4 - 8 jam,4 - 8 jam,4 - 8 jam,4 - 8 jam,Tidak punya mesin cuci,Tidak punya setrika,Tidak,Ya,Tidak,Tidak,Tidak,Tidak,Tidak,3-8 Jam,<2 Jam,3-8 Jam,3-8 Jam,<2 Jam,9-12 Jam,TIdak,Tidak mempunyai kendaraan listrik,Rp. 1.500.001 - Rp 2.000.000,September,3.0


In [6]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 286
Columns: 50


## 2. Remove Columns

These fields were excluded during the original preprocessing because they were considered unsuitable for the modeling workflow, including high-cardinality location data, ranking fields, and free-text responses.

In [7]:
drop_cols = [
    'x2',   # Kota tempat tinggal (terlalu banyak kategori)
    'x25','x26','x27','x28','x29','x30','x31','x32','x33','x34',  # ranking peralatan elektronik
    'x63', 'x59', 'x61', 'x62'   # jawaban teks terkait pengurangan penggunaan listrik
]

df = df.drop(columns=drop_cols, errors='ignore')
df.head()

,Unnamed: 0,x1,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,x17,x18,x19,x20,x21,x22,x23,x24,x35,x36,x37,x38,x39,x40,x41,x42,x43,x44,x45,x46,x47,x48,x49,x50,x51,x52,x53,x54,x55,x56,x57,x58,x60
0,0,Iya,Rumah,4,Sejuk,> 2200 VA,Tidak,lebih dari 20,Tidak ada,Tidak ada,1,Tidak ada,1,Tidak ada,Tidak ada,Tidak ada,Tidak ada,3,1,1,4,Tidak ada,lebih dari 5,1,2 - 4 jam,4 - 8 jam,> 8 jam,> 12 jam,2 - 4 jam,8 - 12 jam,> 4 Kali,> 4 kali,Ya,Ya,Tidak,Tidak,Tidak,Ya,Ya,9-12 Jam,> 12 Jam,3-8 Jam,3-8 Jam,3-8 Jam,9-12 Jam,TIdak,Tidak mempunyai kendaraan listrik,Rp. 1.000.001 - Rp 1.500.000,"Januari, Juni, Juli, Desember"
1,1,Iya,Rumah,4,Sejuk,> 2200 VA,Ya,lebih dari 20,2,1,1,1,Tidak ada,Tidak ada,2,3,2,2,2,2,4,Tidak ada,2,2,4 - 8 jam,4 - 8 jam,4 - 8 jam,8 - 12 jam,8 - 12 jam,4 - 8 jam,2 kali,2 kali,Ya,Ya,Ya,Tidak,Ya,Ya,Tidak,> 12 Jam,3-8 Jam,9-12 Jam,9-12 Jam,3-8 Jam,9-12 Jam,Ya,Tidak mempunyai kendaraan listrik,Rp. 1.500.001 - Rp 2.000.000,Tidak Ada perbedaan signifikan
2,2,Iya,Kos,1,Sejuk,900 VA,Ya,1-5,1,Tidak ada,Tidak ada,1,Tidak ada,Tidak ada,1,1,1,Tidak ada,1,1,1,Tidak ada,Tidak ada,Tidak ada,Tidak punya TV,Tidak punya TV,4 - 8 jam,8 - 12 jam,4 - 8 jam,8 - 12 jam,Tidak punya mesin cuci,Tidak punya setrika,Tidak,Tidak,Tidak,Tidak,Tidak,Tidak,Tidak,9-12 Jam,<2 Jam,<2 Jam,<2 Jam,<2 Jam,<2 Jam,TIdak,Tidak mempunyai kendaraan listrik,< Rp. 500.000,"November, Desember"
3,3,Iya,Rumah,> 5,Panas,> 2200 VA,Ya,lebih dari 20,1,1,1,1,1,Tidak ada,Tidak ada,Tidak ada,Tidak ada,1,2,1,4,Tidak ada,2,1,< 2 jam,< 2 jam,> 8 jam,> 12 jam,Tidak punya komputer,Tidak punya komputer,3 kali,1 kali,Ya,Ya,Tidak,Tidak,Tidak,Tidak,Tidak,9-12 Jam,<2 Jam,<2 Jam,<2 Jam,3-8 Jam,9-12 Jam,Ya,Tidak mempunyai kendaraan listrik,> Rp 2.000.000,Tidak Ada perbedaan signifikan
4,4,Iya,Rumah,5,Panas,900 VA,Ya,lebih dari 20,1,Tidak ada,1,Tidak ada,Tidak ada,2,1,Tidak ada,Tidak ada,lebih dari 5,Tidak ada,2,lebih dari 5,Tidak ada,2,Tidak ada,< 2 jam,< 2 jam,4 - 8 jam,4 - 8 jam,4 - 8 jam,4 - 8 jam,Tidak punya mesin cuci,Tidak punya setrika,Tidak,Ya,Tidak,Tidak,Tidak,Tidak,Tidak,3-8 Jam,<2 Jam,3-8 Jam,3-8 Jam,<2 Jam,9-12 Jam,TIdak,Tidak mempunyai kendaraan listrik,Rp. 1.500.001 - Rp 2.000.000,September


## 3. Define Category Mappings

In [8]:
mapping_durasi_jam = {
    'tidak punya tv': 0, 'tidak punya ac/kipas angin': 0, 'tidak punya komputer': 0,
    'tidak mempunyai kendaraan listrik': 0,
    '1-2 jam': 2, '>4 jam': 4, '3-4 jam': 3,
    '<2 jam': 2, '< 2 jam': 2,
    '2-4 jam': 4, '2 - 4 jam': 4,
    '4-8 jam': 8, '4 - 8 jam': 8,
    '3-8 jam': 8, '3 - 8 jam': 8,
    '8-12 jam': 12, '8 - 12 jam': 12,
    '>8 jam': 12, '> 8 jam': 12,
    '>12 jam': 14, '> 12 jam': 14,
    '9-12 jam': 12
}

mapping_frekuensi_minggu = {
    'tidak punya mesin cuci': 0,
    'tidak punya setrika': 0,
    'tidak mempunyai kendaraan listrik': 0,
    '1 kali': 1, '2 kali': 2, '3 kali': 3, '4 kali': 4,
    '>4 kali': 5, '> 4 kali': 5
}

mapping_lampu = {
    '<2 jam': 2, '< 2 jam': 2,
    '3-8 jam': 8, '3 - 8 jam': 8,
    '9-12 jam': 12, '9 - 12 jam': 12,
    '>12 jam': 15, '> 12 jam': 15
}

mapping_biner = {
    'ya': 1, 'iya': 1, 'Iya': 1,
    'tidak': 0, 'Tidak': 0,
    'sejuk': 0, 'panas': 1
}

mapping_daya_listrik = {
    '450 va': 450, '900 va': 900, '1300 va': 1300,
    '2200 va': 2200, '> 2200 va': 3500, '>2200 va': 3500
}

mapping_jumlah_lampu = {
    'tidak ada': 0,
    '1-5': 3, '6-10': 8, '10-15': 13,
    '16-20': 18, 'lebih dari 20': 22
}

appliance_map = {
    'tidak ada': 0,
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    'lebih dari 5': 6
}

mapping_anggota = {
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '> 5': 6
}

mapping_tempat_tinggal = {
    'rumah': 1, 'apartemen': 2, 'kos': 3, 'kontrakan': 4
}

mapping_tagihan_mean = {
    '< rp 500000': 250000,
    'rp 500000 - rp 1000000': 750000,
    'rp 1000001 - rp 1500000': 1250000,
    'rp 1500001 - rp 2000000': 1750000,
    '> rp 2000000': 2500000
}

In [9]:
kolom_durasi_jam = ['x35','x36','x37','x38','x39','x40','x50','x51','x52','x53','x54','x55','x57']
kolom_frekuensi = ['x41','x42']
kolom_biner = ['x1','x5','x7','x43','x44','x45','x46','x47','x48','x49','x56']
kolom_daya_listrik = ['x6']
kolom_jumlah_lampu = ['x8']
kolom_appliance = ['x9','x10','x11','x12','x13','x14','x15','x16','x17','x18','x19','x20','x21','x22','x23','x24']
kolom_anggota = ['x4']
kolom_tempat = ['x3']
kolom_tagihan = ['x58']

In [10]:
def apply_mapping(df, col, mapping_dict):
    df[col + '_mapped'] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping_dict)
    )
    return df

for col in kolom_durasi_jam:
    df = apply_mapping(df, col, mapping_durasi_jam)

for col in kolom_frekuensi:
    df = apply_mapping(df, col, mapping_frekuensi_minggu)

for col in kolom_biner:
    df = apply_mapping(df, col, mapping_biner)

for col in kolom_daya_listrik:
    df = apply_mapping(df, col, mapping_daya_listrik)

for col in kolom_jumlah_lampu:
    df = apply_mapping(df, col, mapping_jumlah_lampu)

for col in kolom_appliance:
    df = apply_mapping(df, col, appliance_map)

for col in kolom_anggota:
    df = apply_mapping(df, col, mapping_anggota)

for col in kolom_tempat:
    df = apply_mapping(df, col, mapping_tempat_tinggal)

for col in kolom_tagihan:
    df = apply_mapping(df, col, mapping_tagihan_mean)

## 4. Handle Selected Missing Values

In [11]:
# Preserve the original project's selected mode-imputation logic.
for col in ['x3_mapped', 'x57_mapped']:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

binary_cols = ['x45_mapped', 'x46_mapped', 'x47_mapped', 'x48_mapped', 'x49_mapped']

for col in binary_cols:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

In [12]:
mapped_cols = [col for col in df.columns if col.endswith('_mapped')]

missing_counts = df[mapped_cols].isna().sum().sort_values(ascending=False)
missing_counts[missing_counts > 0]

x58_mapped    286
dtype: int64

## 5. Construct the Binary Target

The original project used `x60` to derive the classification target:

- A response containing a month name → `1` (Increase)
- A response containing `tidak ada perbedaan signifikan` → `0` (No Increase)

In [13]:
col = 'x60'
df[col] = df[col].astype(str).str.strip().str.lower()

bulan_list = [
    'januari', 'februari', 'maret', 'april', 'mei', 'juni',
    'juli', 'agustus', 'september', 'oktober', 'november', 'desember'
]

def label_biner(val):
    val = str(val).lower()
    if any(bulan in val for bulan in bulan_list):
        return 1
    elif 'tidak ada perbedaan signifikan' in val:
        return 0
    else:
        return None

df['label_kenaikan'] = df[col].apply(label_biner)

df[[col, 'label_kenaikan']].head()

,x60,label_kenaikan
0,"januari, juni, juli, desember",1
1,tidak ada perbedaan signifikan,0
2,"november, desember",1
3,tidak ada perbedaan signifikan,0
4,september,1


In [14]:
total_naik = df['label_kenaikan'].sum()
total_tidak_naik = df['label_kenaikan'].notna().sum() - total_naik

print(f"Increase (1): {int(total_naik)}")
print(f"No Increase (0): {int(total_tidak_naik)}")
print(f"Unresolved labels: {df['label_kenaikan'].isna().sum()}")

Increase (1): 63
No Increase (0): 223
Unresolved labels: 0


## 6. Assemble Final Features

In [15]:
mapped_cols = [col for col in df.columns if col.endswith('_mapped')]

numeric_original_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

exclude_cols = (
    mapped_cols +
    ['label_kenaikan'] +
    [c for c in df.columns if c.endswith('_clean')]
)

numeric_original_cols = [
    col for col in numeric_original_cols
    if col not in exclude_cols
]

final_feature_cols = mapped_cols + numeric_original_cols

print("Jumlah fitur akhir:", len(final_feature_cols))

Jumlah fitur akhir: 48


In [16]:
df_clean = df.dropna(subset=['label_kenaikan']).copy()

X = df_clean[final_feature_cols].copy()
y = df_clean['label_kenaikan'].astype(int).copy()

print("Final rows:", len(df_clean))
print("Features:", X.shape[1])

Final rows: 286
Features: 48


## 7. Export Processed Dataset

In [17]:
df_mapped = df_clean[final_feature_cols].copy()
df_mapped['label_kenaikan'] = y.values

df_mapped.to_csv("../data/processed_household_survey.csv", index=False)

df_mapped.head()

,x35_mapped,x36_mapped,x37_mapped,x38_mapped,x39_mapped,x40_mapped,x50_mapped,x51_mapped,x52_mapped,x53_mapped,x54_mapped,x55_mapped,x57_mapped,x41_mapped,x42_mapped,x1_mapped,x5_mapped,x7_mapped,x43_mapped,x44_mapped,x45_mapped,x46_mapped,x47_mapped,x48_mapped,x49_mapped,x56_mapped,x6_mapped,x8_mapped,x9_mapped,x10_mapped,x11_mapped,x12_mapped,x13_mapped,x14_mapped,x15_mapped,x16_mapped,x17_mapped,x18_mapped,x19_mapped,x20_mapped,x21_mapped,x22_mapped,x23_mapped,x24_mapped,x4_mapped,x3_mapped,x58_mapped,Unnamed: 0,label_kenaikan
0,4,8,12,14,4,12,12,14,8,8,8,12,0.0,5,5,1,0,0,1,1,0.0,0.0,0.0,1.0,1.0,0,3500,22,0,0,1,0,1,0,0,0,0,3,1,1,4,0,6,1,4,1.0,NaN,0,1
1,8,8,8,12,12,8,14,8,12,12,8,12,0.0,2,2,1,0,1,1,1,1.0,0.0,1.0,1.0,0.0,1,3500,22,2,1,1,1,0,0,2,3,2,2,2,2,4,0,2,2,4,1.0,NaN,1,0
2,0,0,8,12,8,12,12,2,2,2,2,2,0.0,0,0,1,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0,900,3,1,0,0,1,0,0,1,1,1,0,1,1,1,0,0,0,1,3.0,NaN,2,1
3,2,2,12,14,0,0,12,2,2,2,8,12,0.0,3,1,1,1,1,1,1,0.0,0.0,0.0,0.0,0.0,1,3500,22,1,1,1,1,1,0,0,0,0,1,2,1,4,0,2,1,6,1.0,NaN,3,0
4,2,2,8,8,8,8,8,2,8,8,2,12,0.0,0,0,1,1,1,0,1,0.0,0.0,0.0,0.0,0.0,0,900,22,1,0,1,0,0,2,1,0,0,6,0,2,6,0,2,0,5,1.0,NaN,4,1
